In [6]:
# %pip install numpy pandas scikit-learn

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("rawExoPlanetData.csv", comment="#")

df = df.dropna(subset=["hostname"])

grouped = df.groupby('hostname')

In [3]:
num_unique_hosts = df["hostname"].nunique()
print("Unique host names:", num_unique_hosts)

# System-level features:
# - num_stars: number of stars in the host system
# - num_planets: number of planets in the system
# - orbital_period_mean: average orbital period across planets
# - orbit_semi_major_min: closest planetary orbit in the system
# - orbit_semi_major_max: farthest planetary orbit in the system
# - planet_radius_mean: average planet radius in the system
# - planet_radius_std: variation in planet radius within the system
# - planet_mass_mean: average planet mass in the system
# - planet_mass_std: variation in planet mass within the system
# - eccentricity_mean: average orbital eccentricity in the system
#
# Derived features:
# - log_orbital_period_mean: log-scaled average orbital period
# - orbital_span: distance between inner and outer orbit
# - system_compactness: planets per unit orbital span
# - mass_to_radius_ratio: rough proxy for average planetary composition
# - planet_per_star_ratio: number of planets normalized by number of stars
#
# Stellar features:
# - stellar_temp: host star effective temperature
# - stellar_radius: host star radius
# - stellar_mass: host star mass
# - stellar_metallicity: host star metallicity


system_df = grouped.agg(
    num_stars=("sy_snum", "first"),
    num_planets=("sy_pnum", "first"),
    
    orbital_period_mean=("pl_orbper", "mean"),
    inner_orbit=("pl_orbsmax", "min"),
    outer_orbit=("pl_orbsmax", "max"),
    
    planet_radius_mean=("pl_rade", "mean"),
    planet_radius_std=("pl_rade", "std"),
    
    planet_mass_mean=("pl_bmasse", "mean"),
    planet_mass_std=("pl_bmasse", "std"),
    
    eccentricity_mean=("pl_orbeccen", "mean"),
    
    stellar_temp=("st_teff", "first"),
    stellar_radius=("st_rad", "first"),
    stellar_mass=("st_mass", "first"),
    stellar_metallicity=("st_met", "first")
).reset_index()


# Derived features
system_df["log_orbital_period_mean"] = np.log10(system_df["orbital_period_mean"])
system_df["orbital_span"] = system_df["outer_orbit"] - system_df["inner_orbit"]
system_df["system_compactness"] = system_df["num_planets"] / system_df["orbital_span"]
system_df["mass_to_radius_ratio"] = system_df["planet_mass_mean"] / system_df["planet_radius_mean"]
system_df["planet_per_star_ratio"] = system_df["num_planets"] / system_df["num_stars"]



# Replace inf values that can happen from divide-by-zero
system_df = system_df.replace([np.inf, -np.inf], np.nan)

# Quick check
print("Number of grouped systems:", len(system_df))
print("Unique hostnames:", df["hostname"].nunique())
print("Matches:", len(system_df) == df["hostname"].nunique())

system_df.head()

Unique host names: 4590
Number of grouped systems: 4590
Unique hostnames: 4590
Matches: True


,hostname,num_stars,num_planets,orbital_period_mean,inner_orbit,outer_orbit,planet_radius_mean,planet_radius_std,planet_mass_mean,planet_mass_std,eccentricity_mean,stellar_temp,stellar_radius,stellar_mass,stellar_metallicity,log_orbital_period_mean,orbital_span,system_compactness,mass_to_radius_ratio,planet_per_star_ratio
0,11 Com,2,1,323.210000,1.178,1.178,12.20,NaN,4914.898486,NaN,0.23800,4874.0,13.76,2.09,-0.26,2.509485,0.000,NaN,402.860532,0.500000
1,11 UMi,1,1,516.219970,1.530,1.530,12.30,NaN,4684.814200,NaN,0.08000,4213.0,29.79,2.78,-0.02,2.712835,0.000,NaN,380.879203,1.000000
2,14 And,1,1,186.760000,0.775,0.775,13.10,NaN,1131.151301,NaN,0.00000,4888.0,11.55,1.78,-0.21,2.271284,0.000,NaN,86.347428,1.000000
3,14 Her,1,2,8749.434445,2.839,20.000,12.55,0.070711,2669.758619,224.738622,0.44415,5338.0,0.93,0.97,0.43,3.941980,17.161,0.116543,212.729770,2.000000
4,16 Cyg B,3,1,798.500000,1.660,1.660,13.50,NaN,565.737400,NaN,0.68000,5750.0,1.13,1.08,0.06,2.902275,0.000,NaN,41.906474,0.333333


In [12]:
clean_df = system_df.copy()

# single-planet systems will have undefined std values
clean_df.loc[clean_df["num_planets"] == 1, "planet_radius_std"] = 0
clean_df.loc[clean_df["num_planets"] == 1, "planet_mass_std"] = 0

# span = 0 is valid for single-planet systems, but compactness breaks there
clean_df.loc[clean_df["orbital_span"] == 0, "system_compactness"] = 0

# drop rows missing the main features we actually care about
core_cols = [
    "num_stars",
    "num_planets",
    "orbital_period_mean",
    "inner_orbit",
    "outer_orbit",
    "planet_radius_mean",
    "planet_mass_mean",
    "eccentricity_mean",
    "stellar_temp",
    "stellar_radius",
    "stellar_mass",
    "stellar_metallicity"
]

clean_df = clean_df.dropna(subset=core_cols)

# fill remaining engineered-feature NaNs with something simple
fill_cols = [
    "planet_radius_std",
    "planet_mass_std",
    "log_orbital_period_mean",
    "orbital_span",
    "mass_to_radius_ratio",
    "planet_per_star_ratio"
]

for col in fill_cols:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# quick check
print("Original shape:", system_df.shape)
print("Cleaned shape:", clean_df.shape)
print("\nRemaining missing values:")
print(clean_df.isna().sum().sort_values(ascending=False).head(15))
print("\nPreview:")
clean_df.head(10)

clean_df.to_csv('cleanedDataForAnalysis.csv', index=False)

Original shape: (4590, 20)
Cleaned shape: (3663, 20)

Remaining missing values:
hostname               0
num_stars              0
num_planets            0
orbital_period_mean    0
inner_orbit            0
outer_orbit            0
planet_radius_mean     0
planet_radius_std      0
planet_mass_mean       0
planet_mass_std        0
eccentricity_mean      0
stellar_temp           0
stellar_radius         0
stellar_mass           0
stellar_metallicity    0
dtype: int64

Preview:


In [13]:
from sklearn.preprocessing import StandardScaler

# features for clustering - no stellar features
cluster_features = [
    "num_stars",
    "num_planets",
    "log_orbital_period_mean",
    "inner_orbit",
    "outer_orbit",
    "planet_radius_mean",
    "planet_mass_mean",
    "planet_mass_std",
    "eccentricity_mean",
    "orbital_span",
    "planet_per_star_ratio"
]

normalize_df = clean_df[cluster_features].copy()

normalize_df["planet_mass_mean"] = np.log10(normalize_df["planet_mass_mean"])

# scale everything
scaler = StandardScaler()
scaled = scaler.fit_transform(normalize_df)

# back to dataframe for readability
scaled_df = pd.DataFrame(scaled, columns=cluster_features)

# quick sanity checks
print("Shape:", scaled_df.shape)
print("\nMeans (≈0):")
print(scaled_df.mean().round(2))
print("\nStd (≈1):")
print(scaled_df.std().round(2))

print("\nPreview:")
scaled_df.head()

scaled_df.to_csv('normalizedDataForClustering.csv', index=False)

Shape: (3663, 11)

Means (≈0):
num_stars                  0.0
num_planets                0.0
log_orbital_period_mean    0.0
inner_orbit               -0.0
outer_orbit               -0.0
planet_radius_mean         0.0
planet_mass_mean          -0.0
planet_mass_std           -0.0
eccentricity_mean         -0.0
orbital_span               0.0
planet_per_star_ratio      0.0
dtype: float64

Std (≈1):
num_stars                  1.0
num_planets                1.0
log_orbital_period_mean    1.0
inner_orbit                1.0
outer_orbit                1.0
planet_radius_mean         1.0
planet_mass_mean           1.0
planet_mass_std            1.0
eccentricity_mean          1.0
orbital_span               1.0
planet_per_star_ratio      1.0
dtype: float64

Preview:
